In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

products = pd.read_csv('product_info.csv')
reviews = pd.read_csv('reviews_0-250.csv')

print("PRODUCTS shape:", products.shape)
print("\nPRODUCTS columns:", products.columns.tolist())
print("\nREVIEWS shape:", reviews.shape)
print("\nREVIEWS columns:", reviews.columns.tolist())

PRODUCTS shape: (8494, 27)

PRODUCTS columns: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']

REVIEWS shape: (42064, 19)

REVIEWS columns: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']


In [5]:

products_clean = products.drop(columns=[
    'variation_type', 'variation_value', 'variation_desc',
    'value_price_usd', 'sale_price_usd', 'child_count',
    'child_max_price', 'child_min_price'
], errors='ignore')


products_clean = products_clean.dropna(
    subset=['rating', 'price_usd', 'brand_name', 'primary_category']
)

products_clean = products_clean.reset_index(drop=True)

print("Count remaining products:", products_clean.shape[0])
print("\nCheck available categories:")
print(products_clean['primary_category'].unique())
print("\nCreate price range bands:")
print("Min: $", products_clean['price_usd'].min())
print("Max: $", products_clean['price_usd'].max())
print("\nTop 10 Brands:")
print(products_clean['brand_name'].value_counts().head(10))

Count remaining products: 8216

Check available categories:
['Fragrance' 'Bath & Body' 'Mini Size' 'Hair' 'Makeup' 'Skincare'
 'Tools & Brushes' 'Men' 'Gifts']

Create price range bands:
Min: $ 3.0
Max: $ 1900.0

Top 10 Brands:
brand_name
SEPHORA COLLECTION         351
CLINIQUE                   177
Dior                       133
tarte                      126
Kérastase                  107
Bumble and bumble          107
Charlotte Tilbury           98
TOM FORD                    97
Anastasia Beverly Hills     94
NEST New York               92
Name: count, dtype: int64


In [7]:
!pip install plotly --quiet
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.express as px
import pandas as pd

category_brand = products_clean.groupby(
    ['primary_category', 'brand_name']
).size().reset_index(name='product_count')

category_brand = category_brand[category_brand['product_count'] >= 5]

fig = px.sunburst(
    category_brand,
    path=['primary_category', 'brand_name'],
    values='product_count',
    title='Beauty Categories & Top Brands',
    color='product_count',
    color_continuous_scale=[
        '#FFB3C6', '#FF85A1', '#FFC8DD',
        '#CDB4DB', '#A2D2FF', '#BDE0FE'
    ],
)

fig.update_layout(
    title_font_size=22,
    title_x=0.5,
    paper_bgcolor='white',
    height=600
)

fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    products_clean,
    x='price_usd',
    y='rating',
    z='loves_count',
    color='primary_category',
    size='reviews',
    hover_name='product_name',
    title='3D: Price vs Rating vs Loves Count',
    color_discrete_sequence=[
        '#FFB3C6', '#CDB4DB', '#A2D2FF',
        '#BDE0FE', '#FFC8DD', '#FFAFCC',
        '#B5EAD7', '#FF9AA2', '#FFDAC1'
    ],
    labels={
        'price_usd': 'Price (USD)',
        'rating': 'Rating',
        'loves_count': 'Loves Count'
    }
)
fig.update_layout(
    title_font_size=20,
    title_x=0.5,
    paper_bgcolor='white',
    height=650
)

fig.show()

In [ ]:
import plotly.express as px

treemap_data = products_clean.dropna(subset=['secondary_category'])
treemap_data = treemap_data[treemap_data['secondary_category'] != '']

fig = px.treemap(
    treemap_data,
    path=['primary_category', 'secondary_category'],
    values='reviews',
    color='rating',
    title='Product Categories by Reviews & Rating',
    color_continuous_scale=[
        '#FFC8DD', '#FFAFCC', '#CDB4DB',
        '#A2D2FF', '#BDE0FE'
    ],
    hover_data=['rating']
)

fig.update_layout(
    title_font_size=20,
    title_x=0.5,
    paper_bgcolor='white',
    height=600
)

fig.show()

In [ ]:
import plotly.express as px

fig = px.violin(
    products_clean,
    x='primary_category',
    y='rating',
    color='primary_category',
    title='Rating Distribution by Category',
    box=True,
    color_discrete_sequence=[
        '#FFB3C6', '#CDB4DB', '#A2D2FF',
        '#BDE0FE', '#FFC8DD', '#FFAFCC',
        '#B5EAD7', '#FF9AA2', '#FFDAC1'
    ]
)

fig.update_layout(
    title_font_size=20,
    title_x=0.5,
    paper_bgcolor='white',
    height=600,
    xaxis_tickangle=-30,
    showlegend=False
)

fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

products_clean['price_range'] = pd.cut(
    products_clean['price_usd'],
    bins=[0, 25, 50, 100, 200, 1900],
    labels=['$0-25', '$25-50', '$50-100', '$100-200', '$200+']
)

heatmap_data = products_clean.groupby(
    ['primary_category', 'price_range'],
    observed=True
)['rating'].mean().reset_index()

heatmap_pivot = heatmap_data.pivot(
    index='primary_category',
    columns='price_range',
    values='rating'
)

fig = px.imshow(
    heatmap_pivot,
    title='Average Rating by Category & Price Range',
    color_continuous_scale=[
        '#FFC8DD', '#FFAFCC', '#CDB4DB',
        '#A2D2FF', '#BDE0FE'
    ],
    aspect='auto',
    text_auto='.2f'
)

fig.update_layout(
    title_font_size=20,
    title_x=0.5,
    paper_bgcolor='white',
    height=500
)

fig.show()